In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


In [ ]:
FILE_PATH = "datasets/kidney_gene.csv"
N_CLUSTERS = 2           
TOP_N_VAR = 25500         
PCA_VARIANCE = 0.8
LASSO_C = 0.6           

print(f"Loading {FILE_PATH}...")
df = pd.read_csv(FILE_PATH, sep=',', index_col=0)
X_raw = df.T 

In [ ]:

# Log & Filter
X_log = np.log2(X_raw + 1)
variances = X_log.var(axis=0)
top_genes_idx = variances.nlargest(TOP_N_VAR).index
X_filtered = X_log[top_genes_idx]

# Scale
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_filtered), columns=X_filtered.columns)

print(f"Data ready: {X_scaled.shape[0]} samples, {X_scaled.shape[1]} genes (Top 4500 Variance)")


In [ ]:

t0 = time.time()

# PCA
pca = PCA(n_components=PCA_VARIANCE, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Clustering
kmeans_pca = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels_pca = kmeans_pca.fit_predict(X_pca)
sil_pca = silhouette_score(X_pca, labels_pca)

print(f"PCA Results:")
print(f"Components needed: {X_pca.shape[1]}")
print(f"Silhouette Score: {sil_pca:.4f}")
print(f"Time: {time.time() - t0:.2f}s")

t0 = time.time()


In [ ]:

# Generate Pseudo-Labels
kmeans_init = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
pseudo_labels = kmeans_init.fit_predict(X_scaled)

# LASSO Selection
lasso = LogisticRegression(C=LASSO_C, penalty='l1', solver='liblinear', random_state=42)
selector = SelectFromModel(lasso, prefit=False)
selector.fit(X_scaled, pseudo_labels)

# Filter Data
selected_genes = X_scaled.columns[selector.get_support()]
X_lasso = X_scaled[selected_genes]

# Re-Cluster
kmeans_lasso = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels_lasso = kmeans_lasso.fit_predict(X_lasso)
sil_lasso = silhouette_score(X_lasso, labels_lasso)
    


In [ ]:
print(f"LASSO Results:")
print(f"- Genes kept: {len(selected_genes)}")
print(f"- Silhouette Score: {sil_lasso:.4f}")
print(f"- Time: {time.time() - t0:.2f}s")

# PCA on openML

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.impute import SimpleImputer
from scipy.sparse import issparse


In [ ]:
DATASET_ID = 41084 
PCA_VARIANCE = 0.6 # Keep components describing 95% of data

print(f"Fetching OpenML dataset {DATASET_ID}...")
dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
X_raw = dataset.data
y_raw = dataset.target

# 1. Handle Sparse Matrices
if issparse(X_raw):
    X_dense = pd.DataFrame(X_raw.toarray())
else:
    X_dense = pd.DataFrame(X_raw).select_dtypes(include=[np.number])


In [ ]:

imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_dense)

min_val = np.min(X_imputed)
if min_val < 0:
    # negative values detected, likely physics data
    X_processed = X_imputed
else:
   # Data is non-negative (Counts/RNA). Applying Log2(x+1)...
    X_processed = np.log2(X_imputed + 1)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)
try:
    if y_raw is not None:
        le = LabelEncoder()
        y_numeric = le.fit_transform(y_raw)
        n_clusters = len(np.unique(y_numeric))
        print(f"   Target found with {n_clusters} classes.")
    else:
        raise ValueError
except:
    n_clusters = 2
    y_numeric = None
    print("No labels. Running unsupervised (k=2).")


In [ ]:

print("\n--- Baseline (All Features) ---")
kmeans_base = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels_base = kmeans_base.fit_predict(X_scaled)

sil_base = silhouette_score(X_scaled, labels_base)
print(f"Silhouette Score: {sil_base:.4f}")

if y_numeric is not None:
    ari_base = adjusted_rand_score(y_numeric, labels_base)
    print(f"ARI Score:        {ari_base:.4f}")


In [ ]:

print("\n--- PCA Compression ---")
# If float (0.95), it selects components to reach 95% variance
# If int (50), it selects exactly 50 components
pca = PCA(n_components=PCA_VARIANCE, random_state=42)
X_pca = pca.fit_transform(X_scaled)

n_comps = X_pca.shape[1]
print(f"PCA reduced features from {X_scaled.shape[1]} -> {n_comps} components.")

print(f"\n--- Clustering on {n_comps} PCA Components ---")

kmeans_pca = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels_pca = kmeans_pca.fit_predict(X_pca)

# Calculate Silhouette on the REDUCED data (X_pca)
# This measures how tight the clusters are in the new PCA space.
sil_pca = silhouette_score(X_pca, labels_pca)
print(f"Silhouette Score: {sil_pca:.4f}")


In [ ]:

if y_numeric is not None:
    ari_pca = adjusted_rand_score(y_numeric, labels_pca)
    print(f"ARI Score:        {ari_pca:.4f}")
    print(f"Baseline ARI: {ari_base:.4f}")
    print(f"PCA ARI:      {ari_pca:.4f}")
    
